# **Carga del archivo de excel**

In [77]:
import pandas as pd

ruta = "data/raw/Base_Examen_Analista_Datos1.xlsx"
hojas = pd.read_excel(ruta, sheet_name=None)

for nombre, df in hojas.items():
    print(f"--- {nombre} --- {df.shape[0]} filas x {df.shape[1]} columnas")
    print(list(df.columns))
    print()

--- Clientes --- 5005 filas x 10 columnas
['id_cliente', 'edad', 'ciudad', 'antiguedad_meses', 'ingreso_mensual', 'segmento_riesgo', 'cupo_tdc', 'saldo_actual', 'mora_dias', 'producto_principal']

--- Transacciones --- 50000 filas x 6 columnas
['id_tx', 'id_cliente', 'fecha_tx', 'tipo_tx', 'valor_tx', 'categoria']

--- Campañas --- 4000 filas x 7 columnas
['id_cliente', 'campaña', 'fecha_envio', 'canal', 'abrio', 'clic', 'desembolso_libre_destino']

--- Metas_Banco --- 6 filas x 2 columnas
['KPI', 'Meta']



# **PARTE 1 -Calidad de Datos (10 puntos)**

In [78]:
hojas.keys()

dict_keys(['Clientes', 'Transacciones', 'Campañas', 'Metas_Banco'])

In [79]:
nombres_tabla = list(hojas.keys()) # la idea es que tenemos un diccionario con las hojas de excel, acá solo obtengo la lista de nombres
print(nombres_tabla)

['Clientes', 'Transacciones', 'Campañas', 'Metas_Banco']


In [80]:
# Asigno las variables nada mas en el orden que salen en la lista
clientes = hojas[nombres_tabla[0]]
transacciones = hojas[nombres_tabla[1]]
campañas = hojas[nombres_tabla[2]]
metas_banco = hojas[nombres_tabla[3]]

clientes.head()
metas_banco.head()

,KPI,Meta
0,Facturacion,"20,000,000,000.00"
1,Clientes Activos,"3,500.00"
2,Utilizacion,0.45
3,Avances,"3,000,000,000.00"
4,Mora,0.05


## **función resumen_calidad**

In [81]:
# para hacerlo mas sencillo porque son muchas columnas y 4 tablas, creo esta función para simplemente tener un resumen breve de calidad de datos
def resumen_calidad(df, nombre):
    print(f"{nombre}")
    print(f"Filas: {len(df)} | Columnas: {df.shape[1]} | Filas duplicadas: {df.duplicated().sum()}")

    nulos = df.isna().sum()

    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": nulos,
        "% nulos": (nulos / len(df) * 100).round(2),
    })

## **1.1 Clientes** ##

In [82]:
clientes.info()

<class 'pandas.DataFrame'>
RangeIndex: 5005 entries, 0 to 5004
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_cliente          5005 non-null   int64  
 1   edad                5005 non-null   int64  
 2   ciudad              4182 non-null   str    
 3   antiguedad_meses    5005 non-null   int64  
 4   ingreso_mensual     4969 non-null   float64
 5   segmento_riesgo     5005 non-null   str    
 6   cupo_tdc            5005 non-null   int64  
 7   saldo_actual        5005 non-null   int64  
 8   mora_dias           5005 non-null   int64  
 9   producto_principal  5005 non-null   str    
dtypes: float64(1), int64(6), str(3)
memory usage: 391.1 KB


In [83]:
pd.set_option("display.float_format", "{:,.2f}".format)
clientes.describe()



,id_cliente,edad,antiguedad_meses,ingreso_mensual,cupo_tdc,saldo_actual,mora_dias
count,"5,005.00","5,005.00","5,005.00","4,969.00","5,005.00","5,005.00","5,005.00"
mean,"2,498.10",46.73,91.65,"8,104,132.05","5,440,959.04","2,777,744.77",29.51
std,"1,444.79",16.73,51.53,"4,589,592.24","4,483,913.06","3,112,718.05",34.31
min,1.00,18.00,1.00,"1,203,403.00","1,000,000.00",376.00,0.00
25%,"1,247.00",32.00,47.00,"4,655,944.00","2,000,000.00","660,466.00",0.00
50%,"2,498.00",47.00,94.00,"7,972,298.00","4,000,000.00","1,585,964.00",30.00
75%,"3,749.00",61.00,135.00,"11,502,923.00","8,000,000.00","3,851,909.00",60.00
max,"5,000.00",75.00,180.00,"120,000,000.00","100,000,000.00","86,361,962.00",90.00


### *Nulos*

In [84]:
clientes['ciudad'].value_counts(dropna=False)


ciudad
Bogota          918
Bucaramanga     828
Barranquilla    824
NaN             823
Medellin        811
Cali            801
Name: count, dtype: int64

In [85]:
columnas = ['ciudad', 'ingreso_mensual']

pd.DataFrame({
    "nulos": clientes[columnas].isnull().sum(),
    "porcentaje": (clientes[columnas].isnull().mean() * 100).round(2)
})

,nulos,porcentaje
ciudad,823,16.44
ingreso_mensual,36,0.72


In [86]:
clientes[clientes['ingreso_mensual'].isna()]

,id_cliente,edad,ciudad,antiguedad_meses,ingreso_mensual,segmento_riesgo,cupo_tdc,saldo_actual,mora_dias,producto_principal
136,137,25,Bucaramanga,86,NaN,Medio,8000000,2688607,0,Tarjeta
273,274,25,Bucaramanga,58,NaN,Bajo,8000000,1249296,90,Tarjeta
410,411,22,Medellin,102,NaN,Bajo,12000000,8599320,0,Tarjeta
547,548,36,Barranquilla,95,NaN,Medio,4000000,372965,0,Libre Destino
684,685,68,NaN,5,NaN,Alto,8000000,112491,60,Tarjeta
821,822,40,NaN,138,NaN,Alto,4000000,1409220,30,Tarjeta
958,959,36,Bogota,70,NaN,Bajo,8000000,7837831,30,Libre Destino
1095,1096,67,Barranquilla,12,NaN,Bajo,4000000,3579729,30,Libre Destino
1232,1233,28,Bogota,112,NaN,Bajo,8000000,4930121,30,Tarjeta
1369,1370,42,Medellin,63,NaN,Bajo,1000000,581238,30,Libre Destino


**Observación**
No veo un patrón clave respecto a los 36 valores nulos de la columna Ingreso mensual, el cual representa el 0.72% de valores nulos

In [87]:
clientes[clientes['ciudad'].isna()]


,id_cliente,edad,ciudad,antiguedad_meses,ingreso_mensual,segmento_riesgo,cupo_tdc,saldo_actual,mora_dias,producto_principal
3,4,42,NaN,77,"7,350,183.00",Medio,8000000,2147356,0,Libre Destino
4,5,45,NaN,36,"2,691,228.00",Bajo,12000000,4846971,0,Tarjeta
6,7,18,NaN,66,"5,725,567.00",Alto,8000000,2353886,0,Libre Destino
17,18,40,NaN,139,"5,475,728.00",Medio,1000000,782458,0,Tarjeta
18,19,23,NaN,49,"7,765,969.00",Alto,8000000,1528220,0,Tarjeta
...,...,...,...,...,...,...,...,...,...,...
4955,4956,75,NaN,120,"14,887,057.00",Bajo,8000000,6657846,30,Tarjeta
4962,4963,67,NaN,171,"5,497,315.00",Medio,2000000,1402061,30,Tarjeta
4964,4965,70,NaN,172,"5,313,851.00",Bajo,1000000,924982,0,Libre Destino
4971,4972,31,NaN,167,"14,951,110.00",Alto,4000000,2346690,30,Libre Destino


In [88]:
clientes_imputacion = clientes.copy()

clientes_imputacion['ciudad'] = clientes_imputacion['ciudad'].fillna('no_especificada')

clientes_imputacion['ciudad'].value_counts(dropna=False)

ciudad
Bogota             918
Bucaramanga        828
Barranquilla       824
no_especificada    823
Medellin           811
Cali               801
Name: count, dtype: int64

In [89]:
display(resumen_calidad(clientes, 'Clientes'))

Clientes
Filas: 5005 | Columnas: 10 | Filas duplicadas: 5


,tipo,nulos,% nulos
id_cliente,int64,0,0.00
edad,int64,0,0.00
ciudad,str,823,16.44
antiguedad_meses,int64,0,0.00
ingreso_mensual,float64,36,0.72
segmento_riesgo,str,0,0.00
cupo_tdc,int64,0,0.00
saldo_actual,int64,0,0.00
mora_dias,int64,0,0.00
producto_principal,str,0,0.00


### *Duplicados*

In [90]:
clientes[clientes.duplicated()]

,id_cliente,edad,ciudad,antiguedad_meses,ingreso_mensual,segmento_riesgo,cupo_tdc,saldo_actual,mora_dias,producto_principal
5000,99,54,Bogota,110,"5,590,949.00",Alto,1000000,128664,60,Tarjeta
5001,100,37,Barranquilla,180,"6,810,599.00",Medio,4000000,1215980,0,Tarjeta
5002,101,39,Barranquilla,13,"5,330,264.00",Medio,2000000,1344734,0,Tarjeta
5003,102,70,Medellin,117,"6,402,762.00",Bajo,1000000,792971,0,Tarjeta
5004,103,50,Barranquilla,90,"14,653,943.00",Medio,1000000,611444,0,Libre Destino


In [91]:
# Acá simplemente hago el tratamiento de los duplicados, es seguro eliminarlos ya que son exactos, y no perdemos información
clientes_imputacion = clientes_imputacion.drop_duplicates()

clientes_imputacion[clientes_imputacion.duplicated()]

,id_cliente,edad,ciudad,antiguedad_meses,ingreso_mensual,segmento_riesgo,cupo_tdc,saldo_actual,mora_dias,producto_principal


In [92]:
# Ahora miramos si hay clientes que se duplican, de pronto tienen cuentas extra creadas

clientes_imputacion['id_cliente'].duplicated().sum()

np.int64(0)

### *Fechas inconsistentes*

In [93]:
clientes_imputacion['edad'].describe().round(1)

count   5,000.00
mean       46.70
std        16.70
min        18.00
25%        32.00
50%        47.00
75%        61.00
max        75.00
Name: edad, dtype: float64

In [94]:
(clientes_imputacion['antiguedad_meses'] / 12).describe().round(1)

count   5,000.00
mean        7.60
std         4.30
min         0.10
25%         3.90
50%         7.80
75%        11.20
max        15.00
Name: antiguedad_meses, dtype: float64

In [95]:
edad_ingreso = clientes_imputacion['edad'] - (clientes_imputacion['antiguedad_meses'] / 12)

edad_ingreso.head()

0   61.08
1   52.17
2   44.25
3   35.58
4   42.00
dtype: float64

In [96]:
# Agrego rangos de edad para ver si hay clientes que ingresaron al banco siendo menores de edad y cuantos
rangos_edad = pd.cut(
    edad_ingreso[edad_ingreso < 18],
    bins=[0, 10, 14, 18],
    labels=["Menos de 10 años", "10 a 14 años", "14 a 18 años"],
    right=False
)

rangos_edad.value_counts()

14 a 18 años        302
10 a 14 años        231
Menos de 10 años    158
Name: count, dtype: int64

In [97]:
# acá solamente creo una columna nueva que ponga false o true cuando hay clientes que ingresaron al banco siendo menores de edad, esto solo lo veo en caso tal de que quiera en la parte de power bi ya mirar por antiguedad
clientes_imputacion['antiguedad_inconsistente'] = edad_ingreso < 18

clientes_imputacion['antiguedad_inconsistente'].value_counts()


antiguedad_inconsistente
False    4309
True      691
Name: count, dtype: int64

### *Valores negativos o atípicos.*

In [98]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

columnas = {
    "ingreso_mensual": "Ingreso mensual",
    "cupo_tdc": "Cupo tarjeta de crédito",
    "saldo_actual": "Saldo actual",
}

fig = make_subplots(rows=1, cols=3, subplot_titles=list(columnas.values()))

for i, (col, titulo) in enumerate(columnas.items(), start=1):
    fig.add_trace(
        go.Box(
            y=clientes_imputacion[col],
            name=titulo,
            marker_color="#2E86C1",
            boxpoints="outliers",
            customdata=clientes_imputacion["id_cliente"],
            hovertemplate="Cliente %{customdata}<br>%{y:$,.0f}<extra></extra>",
        ),
        row=1, col=i,
    )

fig.update_layout(
    title="Distribución de variables financieras de los clientes",
    showlegend=False,
    height=450,
    template="plotly_white",
)
fig.show()

**Observación** en este diagrama de caja y bigotes simplemente estamos viendo como se distribuyen nuestros datos, y podemos ver fácilmente los valores atípicos

básicamente la raya de la mitad solo nos muestra la mediana para esas tres variables, y los valores que estan por fuera de la caja pero dentro de los bigotes, solamente representan valores "normales" extremos, ya los puntos por fuera son aquellos que si son atípicos, por ejemplo en ingreso mensual tenemos a una persona con un ingreso mensual de 120 millones

In [99]:
mayor_al_cupo = clientes_imputacion[clientes_imputacion["saldo_actual"] > clientes_imputacion["cupo_tdc"]]

print(f"Clientes con saldo mayor al cupo: {len(mayor_al_cupo)}")


Clientes con saldo mayor al cupo: 0


In [100]:
(clientes_imputacion.select_dtypes("number") < 0).sum()

id_cliente          0
edad                0
antiguedad_meses    0
ingreso_mensual     0
cupo_tdc            0
saldo_actual        0
mora_dias           0
dtype: int64

In [102]:
# Miro esos extremoss

extremos = clientes_imputacion[
    (clientes_imputacion["ingreso_mensual"] > 20_000_000) |
    (clientes_imputacion["cupo_tdc"] > 12_000_000) |
    (clientes_imputacion["saldo_actual"] > 20_000_000)
]


extremos

,id_cliente,edad,ciudad,antiguedad_meses,ingreso_mensual,segmento_riesgo,cupo_tdc,saldo_actual,mora_dias,producto_principal,antiguedad_inconsistente
54,55,53,Cali,22,"13,374,152.00",Medio,100000000,28242488,0,Tarjeta,False
76,77,23,Bogota,157,"120,000,000.00",Medio,1000000,813989,90,Tarjeta,True
887,888,64,Bogota,59,"120,000,000.00",Alto,8000000,7541841,90,Libre Destino,False
998,999,74,Cali,168,"3,908,689.00",Medio,100000000,86361962,60,Tarjeta,False
2221,2222,46,Medellin,96,"50,000,000.00",Medio,1000000,499439,90,Libre Destino,False


### *Checando otras variables*

In [105]:
# solo hice este for chiquito para ver que no hubieran nombres raros en las columnas de segmento de riesgo y producto principal

for col in ["segmento_riesgo", "producto_principal"]:
    print(clientes_imputacion[col].value_counts(dropna=False))
    print()

segmento_riesgo
Medio    1739
Bajo     1646
Alto     1615
Name: count, dtype: int64

producto_principal
Tarjeta          3327
Libre Destino    1673
Name: count, dtype: int64



In [106]:
clientes_imputacion.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id_cliente                5000 non-null   int64  
 1   edad                      5000 non-null   int64  
 2   ciudad                    5000 non-null   str    
 3   antiguedad_meses          5000 non-null   int64  
 4   ingreso_mensual           4964 non-null   float64
 5   segmento_riesgo           5000 non-null   str    
 6   cupo_tdc                  5000 non-null   int64  
 7   saldo_actual              5000 non-null   int64  
 8   mora_dias                 5000 non-null   int64  
 9   producto_principal        5000 non-null   str    
 10  antiguedad_inconsistente  5000 non-null   bool   
dtypes: bool(1), float64(1), int64(6), str(3)
memory usage: 395.6 KB


### **<span style='color:red'>Conclusión para tabla "clientes"</span>**

| Filas & Columnas | Valor |
|---|---|
| Filas | 5.005 |
| Columnas | 10 |

**Columnas:** id_cliente, edad, ciudad, antiguedad_meses, ingreso_mensual, segmento_riesgo, cupo_tdc, saldo_actual, mora_dias, producto_principal


<span style="color:green"><b>Valores nulos:</b></span> se identificaron valores faltantes en las columnas `ciudad` e `ingreso_mensual` de la tabla clientes. `ciudad` presenta **823** valores nulos (16,44%), mientras que `ingreso_mensual` tiene **36** registros nulos (0,72%).

1. **Tratamiento propuesto para `ciudad`:** debido a que el porcentaje de valores faltantes es considerable, se reemplazan los nulos con una nueva categoría llamada `no_especificada`. Así se evita eliminar registros importantes y se diferencian los clientes cuya ciudad es desconocida de aquellos que sí tienen este dato.
   **Impacto:** los análisis por ciudad cubren el 83,6% de la base; el resto de los análisis no se ve afectado.

2. **Tratamiento propuesto para `ingreso_mensual`:** al tratarse de una variable financiera personal, y dado que los faltantes solo representan el 0,72% de los registros, no se imputan con media, mediana o moda, para no introducir valores que no corresponden al ingreso real del cliente. Tampoco se reemplazan con 0, ya que distorsionaría los resultados. Se recomienda que el área responsable de la captura complete el dato de estos 36 clientes.
   **Impacto:** bajo; los indicadores basados en ingreso se calculan sobre el 99,28% de la base.

<span style="color:green"><b>Duplicados:</b></span> la tabla clientes tiene **5** registros duplicados exactos. Se verificó además que no existen `id_cliente` repetidos con información distinta. Los duplicados están ubicados en las últimas filas de la base, como un bloque consecutivo, lo que sugiere una doble carga de datos. 
El **tratamiento** es eliminar las copias y conservar la primera aparición.
**Impacto:** sin esta corrección, estos 5 clientes contarían doble en los conteos de clientes, la facturación y cualquier promedio.

<span style="color:green"><b>Valores negativos o atípicos.:</b></span> la tabla de clientes no tiene valores negativos, arriba cuando hice clientes.describe() se puede observar que todos los min son positivos. Pero si tenemos valores atípicos

<span style="color:green"><b>algunas incosistencias:</b></span> la tabla clientes no contiene columnas de fecha, pero se revisó la consistencia temporal entre `edad` y `antiguedad_meses`. Por separado, ambas variables tienen rangos válidos (edad de 18 a 75 años; antigüedad de 1 a 180 meses). Sin embargo, al calcular la edad de ingreso de cada cliente (edad − antigüedad en años), **691 clientes (13,82%)** habrían ingresado antes de los 18 años: 302 entre los 14 y los 18, 231 entre los 10 y los 14, y 158 antes de los 10.

El objetivo del caso es aumentar el uso de la tarjeta de crédito en clientes actuales, por lo que yo asumo que son clientes que ya cuentan con la tarjeta de crédito. Dado que ser mayor de edad es requisito para obtener una tarjeta de crédito, estos registros dejan inferir que simplemente corresponden a un error de captura `antiguedad_meses`.

**Tratamiento:** se conservan los registros, ya que el resto de su información es válida, y se marcan con la columna `antiguedad_inconsistente`. Se excluyen solo de los análisis que dependen de la antigüedad. Se recomienda validar con el área responsable si la antigüedad mide la relación con la entidad o tiempo en que recibió la tarjeta, e incorporar una regla de validación en la la antigüedad en meses.

**Impacto:** sin esta marca, cualquier análisis de comportamiento por antigüedad incluiría un 13,82% de registros no confiables.

<span style="color:green"><b>Valores negativos o atípicos:</b></span> no hay valores negativos. En el `describe()` vi que los máximos de `ingreso_mensual`, `cupo_tdc` y `saldo_actual` estaban muy lejos del resto, así que los revisé con caja y bigotes. Los saldos entre 9 y 12 millones no los considero un problema, porque son coherentes con los cupos más altos (8 y 12 millones).

Los que sí llaman la atención son **5 clientes (0,1%)** totalmente separados del resto, con valores que no cuadran entre sí:

- **77, 888 y 2222:** ingresos de 50 a 120 millones, pero cupos de 1 a 8 millones y 90 días de mora sobre deudas bajas. Probable error en el ingreso.
- **999:** cupo de 100 millones, saldo de 86 millones y 60 días de mora. Probable error en el cupo.
- **55:** cupo de 100 millones con ingreso de 13,4 millones y al día. Es el único caso que podría ser real.

**Supuesto:** tomo `saldo_actual` como lo que el cliente debe en la tarjeta.

**Tratamiento e impacto:** los conservo, porque su efecto en los totales es menor al 1%. Los tengo en cuenta en análisis por cliente, como rankings, donde podrían aparecer en los primeros lugares. Igual recomendaría revisar esto con el area encargada

<span style="color:green"><b>Variables que requieren transformación:</b></span>

- `ciudad`: los nulos se reemplazaron por la categoría `no_especificada`.
- `ingreso_mensual`: aparece como decimal porque contiene vacíos; se mantiene así, ya que los nulos se conservan a propósito.
- `antiguedad_inconsistente`: nueva columna que marca a los clientes que habrían ingresado antes de los 18 años.

## **1.2 transacciones** ##

In [108]:
transacciones_imputacion = transacciones.copy()

display(resumen_calidad(transacciones_imputacion, "Transacciones"))
transacciones_imputacion.head()

Transacciones
Filas: 50000 | Columnas: 6 | Filas duplicadas: 0


,tipo,nulos,% nulos
id_tx,int64,0,0.00
id_cliente,int64,0,0.00
fecha_tx,str,0,0.00
tipo_tx,str,0,0.00
valor_tx,int64,0,0.00
categoria,str,0,0.00


,id_tx,id_cliente,fecha_tx,tipo_tx,valor_tx,categoria
0,1,6056,2026-07-07,RECAUDO,1117407,Digital
1,2,6076,2026-06-16,AVANCE,281164,Retail
2,3,6028,2026-08-07,RECAUDO,3422798,Cajas Éxito
3,4,6150,2025-12-05,AVANCE,3542612,ATM
4,5,6117,2026-03-09,AVANCE,178684,Retail


**Observación inicial** 

1. No tenemos valores Nulos ni duplicados en las variables que tenemos
2. En el enunciado eran 7 campos, y la base de datos no incluye la columna de "Comercio" esta variable esta "missing"
3. Sale un tipo "recaudo" sin embargo en el word este no sale
4. los id de los clientes llegan hasta el 6028...
5. fecha es de tipo str, tocaría convertir en fecha

In [115]:
print("id_tx repetidos:", transacciones_imputacion['id_tx'].duplicated().sum())
print()
print(transacciones_imputacion['tipo_tx'].value_counts(dropna=False))
print()
print(transacciones_imputacion['categoria'].value_counts(dropna=False))
print()

id_tx repetidos: 0

tipo_tx
RECAUDO    24999
COMPRA     12592
AVANCE     12409
Name: count, dtype: int64

categoria
Digital        15931
Cajas Éxito    12205
Retail          9258
ATM             6280
Hogar           3229
Viajes          3096
Banco              1
Name: count, dtype: int64



In [ ]:
# me da curiosidad ver si todos los clientes id de transacciones pertenecen a la tabla de clientes, me parece raro que salgan valores de id como 6117

ids_clientes = clientes_imputacion['id_cliente']
sin_cliente = ~transacciones_imputacion['id_cliente'].isin(ids_clientes)
print(f"Transacciones de clientes que no existen en Clientes: {sin_cliente.sum()} ({sin_cliente.mean() * 100:.2f}%)")
print(f"Rango de id_cliente en Transacciones: {transacciones_imputacion['id_cliente'].min()} a {transacciones_imputacion['id_cliente'].max()}")

Transacciones de clientes que no existen en Clientes: 200 (0.40%)
Rango de id_cliente en Transacciones: 1 a 6198


In [120]:
transacciones_imputacion[sin_cliente].head(4)

,id_tx,id_cliente,fecha_tx,tipo_tx,valor_tx,categoria
0,1,6056,2026-07-07,RECAUDO,1117407,Digital
1,2,6076,2026-06-16,AVANCE,281164,Retail
2,3,6028,2026-08-07,RECAUDO,3422798,Cajas Éxito
3,4,6150,2025-12-05,AVANCE,3542612,ATM


In [ ]:
# simplemente guardo esas transacciones de clientes que no tenemos en nuestra base de datos en una variable como para excluirlas de la principal

transacciones_clientes_no_existentes = transacciones_imputacion[sin_cliente]

transacciones_imputacion = transacciones_imputacion[~sin_cliente]

print(f"Transacciones excluidas: {len(transacciones_clientes_no_existentes)}")
print(f"Transacciones que quedan: {len(transacciones_imputacion):,}")

Transacciones excluidas: 200
Transacciones que quedan: 49,800


In [ ]:
# ahora miro los clientes que no tienen transacciones

clientes_sin_transacciones = ~clientes_imputacion['id_cliente'].isin(transacciones_imputacion['id_cliente'])

clientes_imputacion[clientes_sin_transacciones]

,id_cliente,edad,ciudad,antiguedad_meses,ingreso_mensual,segmento_riesgo,cupo_tdc,saldo_actual,mora_dias,producto_principal,antiguedad_inconsistente
2289,2290,48,Barranquilla,156,"2,823,089.00",Medio,4000000,2753686,30,Tarjeta,False


In [126]:
transacciones_imputacion.info()

<class 'pandas.DataFrame'>
RangeIndex: 49800 entries, 200 to 49999
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id_tx       49800 non-null  int64
 1   id_cliente  49800 non-null  int64
 2   fecha_tx    49800 non-null  str  
 3   tipo_tx     49800 non-null  str  
 4   valor_tx    49800 non-null  int64
 5   categoria   49800 non-null  str  
dtypes: int64(3), str(3)
memory usage: 2.3 MB


### *tipo_tx = Recaudo*

In [ ]:
# simplemente hago un crosstab para ver que categorias hay para ese tipo de transacciones recaudo

pd.crosstab(transacciones_imputacion["categoria"], transacciones_imputacion["tipo_tx"])

tipo_tx,AVANCE,COMPRA,RECAUDO
categoria,,,
ATM,6250,0,0
Cajas Éxito,0,0,12127
Digital,0,3127,12765
Hogar,0,3220,0
Retail,6104,3117,0
Viajes,0,3090,0


In [131]:
transacciones_imputacion["tipo_tx"] = transacciones_imputacion["tipo_tx"].replace({"RECAUDO": "PAGO"})

transacciones_imputacion["tipo_tx"].value_counts(dropna=False)

tipo_tx
PAGO      24892
COMPRA    12554
AVANCE    12354
Name: count, dtype: int64

In [132]:
fechas = pd.to_datetime(transacciones_imputacion["fecha_tx"])
fechas.describe()

count                         49800
mean     2026-04-01 11:06:39.036144
min             2025-10-01 00:00:00
25%             2025-12-31 00:00:00
50%             2026-04-01 00:00:00
75%             2026-07-01 00:00:00
max             2026-10-01 00:00:00
Name: fecha_tx, dtype: object

In [133]:
transacciones_imputacion['valor_tx'].describe()

count      49,800.00
mean    2,498,292.35
std     1,444,382.33
min        10,181.00
25%     1,250,907.50
50%     2,497,185.50
75%     3,748,335.00
max     4,999,999.00
Name: valor_tx, dtype: float64

### **<span style='color:red'>Conclusión para tabla "transacciones"</span>**

| Filas & Columnas | Valor | Nuevo Valor
|---|---|---|
| Filas | 50000 | 49999
| Columnas | 6 |

**Columnas:** id_tx, id_cliente, fecha_tx, tipo_tx, valor_tx, categoria

<span style="color:green"><b>Todos los clientes de transacciones estan en nuestra tabla de clientes?</b></span> como principal nota, veo que no todas las transacciones pertenecen a un cliente de nuestra tabla de clientes. Pero hay 200 personas id de cliente que tienen transacciones pero no salen en nuestra tabla de clientes, esto recomendaría ya mirarlo en un análisis mas exhaustivo, pero por ahora las voy a eliminar para poder hacer el ejercicio

<span style="color:green"><b>Clientes sin transacciones:</b></span> Bueno, respondiendo el punto 4 de calidad de datos, solo tenemos 1 cliente de los 5000 que no tiene transacciones, es el cliente con id 2290. Ese cliente debe a la tarjeta, y tiene una mora de 30 días, podemos inferir que como su ingreso_mensual se acerca mucho a lo que debe, esta persona dejó de pagar y usar la tarjeta? porque no presenta transacciones.
**Impacto:** mínimo en los totales, pero es un caso relevante para el análisis de señales de abandono.

<span style="color:green"><b>Transformación de la variable `tipo_txt` -> RECAUDO</b></span> Simplemente para tener consistencia con el documento word, lo cambié a "PAGO"

<span style="color:green"><b>Fechas:</b></span> todas las fechas de `fecha_tx` son validas. Van desde octubre de 2025 a octubre de 2026, sin fechas futuras. Octubre de 2026 solo tiene datos del día 1.

<span style="color:green"><b>Valores:</b></span> `valor_tx` no presenta negativos, ceros ni atípicos.



## **1.3 campañas** ##

In [134]:
campañas_imputacion = campañas.copy()

display(resumen_calidad(campañas_imputacion, "Campañas"))
campañas_imputacion.head()

Campañas
Filas: 4000 | Columnas: 7 | Filas duplicadas: 0


,tipo,nulos,% nulos
id_cliente,int64,0,0.00
campaña,str,0,0.00
fecha_envio,str,0,0.00
canal,str,996,24.90
abrio,int64,0,0.00
clic,int64,0,0.00
desembolso_libre_destino,int64,0,0.00


,id_cliente,campaña,fecha_envio,canal,abrio,clic,desembolso_libre_destino
0,393,Tarjeta Activa,2026-01-05,Email,0,1,0
1,4877,Extracupo,2026-01-26,WhatsApp,0,1,0
2,3208,Libre Destino,2026-04-20,Email,0,1,0
3,4895,Libre Destino,2026-02-15,SMS,0,1,0
4,2078,Libre Destino,2026-04-25,SMS,0,1,0


**Observación inicial** ya de primera mano noto que hay clientes que hacen clic pero no abrieron el mensaje. 

In [135]:
campaña_sin_abrir = (campañas_imputacion['abrio'] == 0) & (campañas_imputacion['clic'] > 0)

campaña_sin_abrir.value_counts()

False    2973
True     1027
Name: count, dtype: int64

In [137]:
campañas_imputacion[campaña_sin_abrir]["canal"].value_counts(dropna=False)

canal
WhatsApp    281
SMS         256
Email       252
NaN         238
Name: count, dtype: int64

**observación** no hay un canal que muestre un patron claro de que sea un problema de medición que los clientes puedan hacer clic. Están distribuidos a ojo de forma proporcional.

In [ ]:
# como no veo que sea algun patron o error con las campañas y que posiblememente solo haya sido un error de registro, imputo esos Abrio = 0 que tienen clic = 1 con 1
campañas_imputacion.loc[campaña_sin_abrir, "abrio"] = 1


False    2973
True     1027
Name: count, dtype: int64

In [142]:
print(campañas_imputacion["campaña"].value_counts(), "\n")
print(campañas_imputacion["canal"].value_counts(dropna=False), "\n")

campaña
Extracupo         1378
Tarjeta Activa    1339
Libre Destino     1283
Name: count, dtype: int64 

canal
WhatsApp    1037
Email       1001
NaN          996
SMS          966
Name: count, dtype: int64 



In [146]:
# imputo los nulos de canal que son 996 con no_especificada

campañas_imputacion['canal'] = campañas_imputacion['canal'].fillna('no_especificada')
print(campañas_imputacion["canal"].value_counts(dropna=False), "\n")

canal
WhatsApp           1037
Email              1001
no_especificada     996
SMS                 966
Name: count, dtype: int64 



In [ ]:
pd.crosstab(
    campañas_imputacion['clic'],
    campañas_imputacion['desembolso_libre_destino'] > 0,
    colnames=['hubo_desembolso']
)

hubo_desembolso,False,True
clic,,
0,1791,261
1,1673,275


**Observación:** hice este cruce para ver si hay desembolsos sin clic y como se relacionan ambas variables. La tasa de desembolso es muy parecida entre quienes hicieron clic (14,1%) y quienes no (12,7%), lo que sugiere que el clic en el mensaje no está asociado a más desembolsos. Además, 261 clientes recibieron el desembolso sin hacer clic, seguro lo solicitaron por otro canal.


In [148]:
print(pd.to_datetime(campañas_imputacion['fecha_envio']).describe(), "\n")



count                          4000
mean     2026-05-16 15:11:31.200000
min             2026-01-01 00:00:00
25%             2026-03-11 00:00:00
50%             2026-05-16 00:00:00
75%             2026-07-21 00:00:00
max             2026-09-28 00:00:00
Name: fecha_envio, dtype: object 



### **<span style='color:red'>Conclusión para tabla "campañas"</span>**

| Filas & Columnas | Valor | 
|---|---|
| Filas | 4000 | 
| Columnas | 7 |

**Columnas:** id_cliente, campaña, fecha_envio, canal, abrio, clic, desembolso_libre_destino

<span style="color:green"><b>Resumen de calidad – Campañas:</b></span> no tenemos duplicados. Las campañas (Tarjeta Activa, Extracupo y Libre Destino) no presentan inconsistencias de escritura, y las fechas de envío son válidas.

- **Nulos en `canal`:** 996 registros (24,9%), repartidos de forma similar a los demás canales. Se reemplazan por `no_especificado`.
- **Clic sin apertura:** 1.027 registros (25,7%). Se corrige `abrio` a 1 cuando `clic` es 1.
- **Desembolsos sin clic:** 261 registros, el cliente opto por otro canal para hacer el desembolso, seguramente.

<span style="color:green"><b>Variables que requieren transformación:</b></span>

- `canal`: nulos reemplazados por `no_especificado`.
- `abrio`: corregido a 1 cuando hay clic.

# **Archivos SQL**

In [155]:
# Guardo mis tablas limpias en csv para poder hacer la parte de SQL

import os
os.makedirs("data/clean", exist_ok=True)

clientes_imputacion.to_csv("data/clean/clientes_imputacion.csv", index=False)
campañas_imputacion.to_csv("data/clean/campanas_imputacion.csv", index=False)
transacciones_imputacion.to_csv("data/clean/transacciones_imputacion.csv", index=False)
metas_banco.to_csv("data/clean/metas_banco.csv", index=False)

# **PARTE 2 - SQL (30 puntos)**

Acá solo uso las tablas limpias con pandas de la parte 1 y la facturación corresponde a los avances y la suma de las compras


In [ ]:
import sqlite3 as sql

con = sql.connect(":memory:")  # con memory la base se crea en mi memoria

clientes_imputacion.to_sql("clientes", con, index=False, if_exists="replace")
transacciones_imputacion.to_sql("transacciones", con, index=False, if_exists="replace")
campañas_imputacion.to_sql("campanas", con, index=False, if_exists="replace")
metas_banco.to_sql("metas", con, index=False, if_exists="replace")

pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", con)

,name
0,clientes
1,transacciones
2,campanas
3,metas


### **Pregunta 1: Calcule por cliente (Total compras, Total avances, Total pagos, Facturación total)**


In [166]:
pregunta_1 = """
SELECT
    id_cliente,
    SUM(CASE WHEN tipo_tx = 'COMPRA' THEN valor_tx ELSE 0 END) AS total_compras,
    SUM(CASE WHEN tipo_tx = 'AVANCE' THEN valor_tx ELSE 0 END) AS total_avances,
    SUM(CASE WHEN tipo_tx = 'PAGO'   THEN valor_tx ELSE 0 END) AS total_pagos,
    SUM(CASE WHEN tipo_tx IN ('COMPRA', 'AVANCE') THEN valor_tx ELSE 0 END) AS facturacion_total
FROM transacciones
GROUP BY id_cliente
ORDER BY id_cliente
"""

pd.read_sql(pregunta_1, con)

,id_cliente,total_compras,total_avances,total_pagos,facturacion_total
0,1,8331974,1522625,13349269,9854599
1,2,11400372,11271106,15464570,22671478
2,3,6263097,3747440,9453449,10010537
3,4,8331628,5378457,22847548,13710085
4,5,9923413,4497619,1001563,14421032
...,...,...,...,...,...
4994,4996,9712146,21324531,8131851,31036677
4995,4997,1518971,6488627,23883757,8007598
4996,4998,0,4407343,17281173,4407343
4997,4999,4185526,9220575,16307403,13406101


- Acá es importante decir que el cliente con id 2290 no ha hecho transacciones, es el único, por ende, no hago ningún join.

### **Pregunta 2: Identifique el Top 20 de clientes por facturación.**


In [163]:
pregunta_2 = """
SELECT 
    c.id_cliente,
    c.segmento_riesgo,
    c.cupo_tdc,
    c.ciudad,
    SUM(t.valor_tx) AS facturacion_total
FROM clientes c 
INNER JOIN transacciones t on c.id_cliente = t.id_cliente
WHERE t.tipo_tx IN ('COMPRA', 'AVANCE')
GROUP BY c.id_cliente, c.segmento_riesgo, c.cupo_tdc, c.ciudad
ORDER BY facturacion_total DESC
LIMIT 20

"""
pd.read_sql(pregunta_2,con)

,id_cliente,segmento_riesgo,cupo_tdc,ciudad,facturacion_total
0,3777,Bajo,4000000,Bogota,39726071
1,4844,Alto,2000000,Bucaramanga,38807337
2,1961,Medio,2000000,Barranquilla,36424331
3,3673,Alto,1000000,Medellin,35920288
4,2399,Alto,2000000,Cali,35637405
5,2679,Medio,1000000,no_especificada,35505927
6,3307,Bajo,12000000,Medellin,34895674
7,3069,Medio,8000000,Barranquilla,34158736
8,3267,Medio,2000000,Barranquilla,33964305
9,653,Alto,1000000,Bucaramanga,33864344


### **Pregunta 3: Calcule para cada segmento de riesgo: (Clientes, Facturación, Ticket promedio)**


In [171]:
pregunta_3 = """

SELECT
    c.segmento_riesgo,
    COUNT(DISTINCT c.id_cliente) AS clientes,
    SUM(t.valor_tx) AS facturacion,
    AVG(t.valor_tx) AS ticket_promedio
FROM clientes c
LEFT JOIN transacciones t ON c.id_cliente = t.id_cliente
WHERE t.tipo_tx IN ('COMPRA', 'AVANCE')
GROUP BY c.segmento_riesgo


"""
pd.read_sql(pregunta_3,con)

,segmento_riesgo,clientes,facturacion,ticket_promedio
0,Alto,1604,20256587443,"2,512,912.47"
1,Bajo,1634,20193966620,"2,480,221.89"
2,Medio,1731,21817237657,"2,506,288.07"


- El ticket promedio es el promedio del valor de las compras y avances de cada segmento. En los tres es muy parecido (alrededor de 2,5 millones por transacción), el nivel de riesgo no determina cuanto gastan los clientes cuando usan la tarjeta
- La facturación también es similar entre segmentos. El segmento Medio factura un poco más, pero es porque tiene un tris más clientes, no porque gasten más.
- Al sumar los clientes de los tres segmentos da 4.969, y no 4.999. Los 30 que faltan tienen transacciones, pero solo pagos: están pagando su deuda pero ya no usan la tarjeta.

### **Pregunta 4: Construya una segmentación: (Heavy users, Medium Users, Low Users)**


In [177]:
pregunta_4 = """
WITH facturacion_cliente AS(
    SELECT
        c.id_cliente,
        SUM(CASE WHEN t.tipo_tx IN ('COMPRA', 'AVANCE') THEN t.valor_tx ELSE 0 END) AS facturacion_total
    FROM clientes c
    LEFT JOIN transacciones t ON c.id_cliente = t.id_cliente
    GROUP BY c.id_cliente
),

grupo_3 AS (
    SELECT
        id_cliente,
        facturacion_total,
        NTILE(3) OVER (ORDER BY facturacion_total DESC) AS grupo
    FROM facturacion_cliente   
)

SELECT
    id_cliente,
    facturacion_total,
    CASE grupo
        WHEN 1 THEN 'Heavy Users'
        WHEN 2 THEN 'Medium Users'
        WHEN 3 THEN 'Low Users'
    END AS segmentacion
FROM grupo_3
"""
segmentacion = pd.read_sql(pregunta_4, con)
segmentacion


,id_cliente,facturacion_total,segmentacion
0,3777,39726071,Heavy Users
1,4844,38807337,Heavy Users
2,1961,36424331,Heavy Users
3,3673,35920288,Heavy Users
4,2399,35637405,Heavy Users
...,...,...,...
4995,4595,0,Low Users
4996,4651,0,Low Users
4997,4658,0,Low Users
4998,4684,0,Low Users


In [178]:
segmentacion["segmentacion"].value_counts()

segmentacion
Heavy Users     1667
Medium Users    1667
Low Users       1666
Name: count, dtype: int64

In [184]:
segmentacion.to_sql('segmentacion', con, index=False, if_exists='replace')

tabla = pd.read_sql("""
SELECT 
    segmentacion,
    COUNT(*) AS clientes,
    SUM(facturacion_total) AS facturacion_total,
    AVG(facturacion_total) AS facturacion_promedio,
    MIN(facturacion_total) AS facturacion_min,
    MAX(facturacion_total) AS facturacion_max
FROM segmentacion
GROUP BY segmentacion
""",con)
 # hago pd.read_sql porque necesita dos parametros, la consulta SQL y la conexion que es la base de datos, asi me agarra segmentación
display(tabla)


,segmentacion,clientes,facturacion_total,facturacion_promedio,facturacion_min,facturacion_max
0,Heavy Users,1667,32903904652,"19,738,395.11",14716978,39726071
1,Low Users,1666,9607639996,"5,766,890.75",0,9152419
2,Medium Users,1667,19756247072,"11,851,377.97",9152486,14712630


**Metodología:** segmenté a los clientes por su facturación total del año (compras + avances), porque es lo que el banco quiere aumentar y porque, como vimos en la pregunta 3, el segmento de riesgo no es muy determinante. 

Usé `WITH` para crear tablas temporales (CTE) y resolver el problema por pasos dentro de una misma consulta:

1. Primero calculé la facturación total de cada cliente. Usé un `LEFT JOIN` desde clientes para incluir también a los que no facturaron nada, que solo es 1.
2. Después, con `NTILE(3)`, ordené a los clientes de mayor a menor facturación y los dividí en tres grupos del mismo tamaño.
3. Al final le asigné a cada grupo su nombre: Heavy, Medium y Low Users.

Elegí dividirlos en tres partes iguales por temas de tiempo, ya que un análisis mas exhaustivo me hubiera ayudado a determinar umbrales, sino de la misma distribución de los datos. Cada grupo quedó con unos 1.667 clientes.


### **Pregunta 5: Identifique clientes con: Más de 60 días de mora y Facturación superior al promedio.**


In [185]:
pregunta_5 = """
   WITH consulta_cliente_mora AS (
        SELECT 
            c.id_cliente,
            c.mora_dias,
            SUM(CASE WHEN t.tipo_tx IN ('COMPRA','AVANCE') THEN t.valor_tx ELSE 0 END) AS facturacion_total
        FROM clientes c
        LEFT JOIN transacciones t ON c.id_cliente = t.id_cliente
        GROUP BY c.id_cliente, c.mora_dias

   ) 
   
   SELECT *
   FROM consulta_cliente_mora
   WHERE mora_dias > 60
    AND facturacion_total > (SELECT AVG(facturacion_total) FROM consulta_cliente_mora)
ORDER BY facturacion_total DESC
"""
pd.read_sql(pregunta_5, con) 

,id_cliente,mora_dias,facturacion_total
0,3628,90,33168919
1,87,90,32216305
2,3548,90,32079236
3,556,90,31903408
4,469,90,31185084
...,...,...,...
360,1258,90,12559775
361,3007,90,12548859
362,2622,90,12489151
363,1119,90,12461340


### **Pregunta 6: Compare(Compras últimos 3 meses, Compras primeros 3 meses) e indentifique crecimiento**


In [192]:
pregunta_6 = """
 WITH periodos_compra AS(
    SELECT 
        SUM(CASE WHEN fecha_tx >='2025-10-01' AND fecha_tx < '2026-01-01' THEN valor_tx ELSE 0 END ) AS primeros_3meses,
        SUM(CASE WHEN fecha_tx >='2026-07-01' AND fecha_tx < '2026-10-01' THEN valor_tx ELSE 0 END ) AS ultimos_3meses
    FROM transacciones
    WHERE tipo_tx = 'COMPRA'
 )
SELECT
    primeros_3meses,
    ultimos_3meses,
    (ultimos_3meses - primeros_3meses) *100.0 / primeros_3meses AS crecimiento
FROM periodos_compra

"""

pd.read_sql(pregunta_6,con)

,primeros_3meses,ultimos_3meses,crecimiento
0,7810952038,7830767954,0.25


Entre los primeros 3 meses (octubre a diciembre de 2025) y los últimos 3 meses completos (julio a septiembre de 2026), las compras pasaron de 7.811 a 7.831 millones, un crecimiento de solo 0,25%. Las compras están prácticamente estancadas, aun con las campañas que el banco realizó durante 2026. Octubre de 2026 no se incluye porque solo tiene un día de datos. Esto me dice que no hay un problema con la estacionalidad

### **Pregunta 7: Construya un ranking de clientes por: facturacion/cupo**


In [194]:
pregunta_7 = """
WITH facturacion_cupo AS (
    SELECT
        c.id_cliente,
        c.cupo_tdc AS cupo,
        SUM(CASE WHEN t.tipo_tx IN ('COMPRA', 'AVANCE') THEN t.valor_tx ELSE 0 END) AS facturacion_total
    FROM clientes c
    LEFT JOIN transacciones t ON c.id_cliente = t.id_cliente
    GROUP BY c.id_cliente, c.cupo_tdc
)
SELECT
    RANK() OVER (ORDER BY facturacion_total * 1.0 / cupo DESC) AS ranking,
    id_cliente,
    cupo,
    facturacion_total,
    facturacion_total * 1.0 / cupo AS facturacion_cupo
FROM facturacion_cupo
ORDER BY ranking
"""

ranking = pd.read_sql(pregunta_7, con)
ranking

,ranking,id_cliente,cupo,facturacion_total,facturacion_cupo
0,1,3673,1000000,35920288,35.92
1,2,2679,1000000,35505927,35.51
2,3,653,1000000,33864344,33.86
3,4,932,1000000,33429367,33.43
4,5,3628,1000000,33168919,33.17
...,...,...,...,...,...
4995,4970,4595,1000000,0,0.00
4996,4970,4651,1000000,0,0.00
4997,4970,4658,1000000,0,0.00
4998,4970,4684,2000000,0,0.00


**Observaición** Los primeros lugares del ranking son clientes con cupo de 1 millón que facturaron cerca de 36 veces su cupo en el año (unos 3 millones al mes). Esto no es consistente con el funcionamiento de una tarjeta de crédito y sugiere que, en esta base, el cupo no limita el valor de las transacciones (hay transacciones de hasta 5 millones con cupos de 1 millón). Los 31 clientes sin facturación comparten el último puesto (4.970).

Si los datos fueran correctos, los clientes de cupo bajo con alto uso serían candidatos naturales para la campaña de Extracupo.

### **Pregunta 8: Identifique clientes: Campañeados - No campañeados, y compare comportamiento.**


### **Pregunta 9: Construya una consulta para identificar señales tempranas de abandono.**


### **Pregunta 10: Construya una consulta para identificar señales tempranas de abandono.**


In [195]:
segmentacion.to_csv("data/clean/segmentacion.csv", index=False)

# **PARTE 4 - Análisis Analítico (20 puntos)**

### **1 ¿Cuáles son los tres hallazgos más relevantes?**

1. **La facturación está por debajo de la meta y no está creciendo.** En el último trimestre completo (julio a septiembre de 2026) la facturación fue de 15.662 millones, un 21,7% por debajo de la meta de 20.000 millones. Además, las compras de los últimos 3 meses crecieron solo un 0,25% frente a los primeros 3 meses del periodo: están prácticamente estancadas. Y de todo esto, vimos también que los primeros 3 meses no tenía campañas puesto que estás iniciaron en el año 2026.
2. **La mora es muy alta.** El 31,6% del saldo tiene más de 30 días de atraso, frente a una meta del 5%. Es más de seis veces lo esperado, y es el indicador más alejado de su meta.
3. **Los avances pesan mucho más** Según las metas, los avances deberían ser cerca del 15% de la facturación, pero en los datos representan casi la mitad. Los clientes están usando la tarjeta tanto para sacar efectivo como para comprar.

### 2. ¿Qué segmentos generan más valor y por qué?

Los **Heavy Users**. Son un tercio de los clientes (1.667), pero generan el **52,8% de la facturación**. Facturan más de 14,7 millones al año, y en promedio 3,4 veces más que un Low User.

Lo interesante es que el ticket promedio (unos 2,5 millones por transacción) es muy parecido en todos los segmentos. Es decir, los Heavy Users no gastan más en cada compra, sino que **usan la tarjeta más veces**. Para aumentar la facturación, la clave está en la frecuencia de uso, no en el monto.

En cambio, el segmento de riesgo (Bajo, Medio y Alto) no diferencia el valor: los tres facturan casi lo mismo, entre 20.000 y 21.800 millones al año.

### 3. ¿Qué segmentos generan más riesgo?

La mora no se concentra en un solo segmento: el porcentaje del saldo con más de 30 días de atraso está entre el 30% y el 33% en los tres segmentos de uso, con los Low Users ligeramente por encima (32,9%). Esto indica que el riesgo es un problema de **toda la cartera**, no de un grupo específico.

Además, el segmento de riesgo asignado por el banco tampoco muestra diferencias claras en el comportamiento, lo que sugiere que esa clasificación no está reflejando el riesgo real de los clientes.

### 4. ¿Qué comportamientos le preocupan?

- **El uso alto de avances**, porque el efectivo suele asociarse a necesidades de liquidez, y combinado con la mora alta es una señal de riesgo.
- **Los clientes que dejaron de usar la tarjeta**: 30 clientes solo hacen pagos y 1 no tiene ninguna transacción. Siguen siendo clientes, pero ya no generan facturación.
- **Las campañas no parecen estar funcionando**: la tasa de desembolso es casi igual entre quienes hicieron clic (14,1%) y quienes no (12,7%), y las compras no crecieron durante el año, a pesar de las campañas de 2026.

### 5. ¿Qué tendencias observa?

- **Estancamiento:** compras y avances se mantienen alrededor de 2.500 millones al mes durante todo el año, sin una tendencia de crecimiento.
- **Concentración del valor:** un tercio de los clientes genera más de la mitad de la facturación.
- **Oportunidad en Libre Destino:** el 33,5% de los clientes tiene como producto principal un crédito de Libre Destino, no la tarjeta. Ya son clientes del banco, pero la tarjeta no es su producto principal.

### 6. ¿Qué anomalías encontró?

- **5 clientes con valores incoherentes:** ingresos de 50 a 120 millones con cupos de 1 millón y 90 días de mora, y dos cupos de 100 millones, uno de ellos 25 veces el ingreso del cliente.
- **Facturación muy superior al cupo:** clientes con cupo de 1 millón facturaron hasta 36 veces su cupo en el año, lo que sugiere que el cupo no está limitando las transacciones.
- **691 clientes (13,8%)** con una antigüedad que implica haber ingresado antes de los 18 años.
- **200 transacciones** de clientes que no existen en la base de clientes.
- **1.027 campañas** con clic pero sin apertura del mensaje.
- **Desembolsos de Libre Destino** registrados en campañas de Tarjeta Activa y Extracupo, por lo que no se pueden atribuir a una campaña específica.
- **Los 36 clientes sin ingreso** 

### 7. ¿Qué análisis adicional haría si tuviera una semana más?

Con más tiempo, lo primero que haría es un EDA más completo. En esta prueba me enfoqué en la calidad de los datos y en responder las preguntas, pero me habría gustado explorar más a fondo cómo se relacionan las variables entre sí, por ejemplo, si la edad, la antigüedad o el cupo influyen en cuánto factura un cliente.

También terminaría de profundizar en las campañas y en el abandono. con las señales de abandono (inactividad, caída en el uso y mora), me gustaría construir un modelo predictivo que ayude a identificar a tiempo a los clientes que están por dejar la tarjeta.

Además, haría tres cosas más:

- **Validar mis supuestos con el negocio**, como el periodo de las metas, la definición del saldo y qué significa exactamente RECAUDO.
- **Mejorar la segmentación**, combinando cuánto factura cada cliente con qué tan seguido usa la tarjeta y cuándo fue la última vez, en lugar de usar solo la facturación.
- **Analizar la rentabilidad**